[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/C.%20Transportation%2C%20Routing%20%26%20Freight%20Management%20%28The%20Physical%20Internet%29/074.%20The%20VRP%20with%20Backhauls%20%28VRPB%29/P74-Tier-9_executed.ipynb)

---


# 74. The Vehicle Routing Problem with Backhauls

## Tier 9: QUBO modeling and the Quantum Approximate Optimization Algorithm

This notebook introduces a QUBO-based modeling route for the Vehicle Routing Problem with Backhauls (VRPB), followed by the main ideas behind QAOA. It builds a small, inspectable quadratic model and describes the resources a QAOA circuit would require. It does not run a quantum circuit or claim a quantum speedup.

### Learning objectives

By the end of the notebook, you should be able to:

- distinguish delivery customers, pickup customers, vehicles, and depot nodes in a VRPB instance;
- interpret binary arc variables and quadratic penalty terms;
- explain how a QUBO objective can be represented by a cost Hamiltonian in QAOA;
- identify which constraints are represented in the illustrative QUBO and which remain outside it;
- distinguish a circuit resource description from an executed quantum optimization experiment.

### Method and scope

The method follows the QUBO-to-QAOA sequence: define binary routing variables, form a quadratic objective with penalty terms, describe a parameterized QAOA circuit, and discuss what an optimization run would require. The concrete instance has four delivery customers, three pickup customers, and two modeled vehicles.

The QUBO constructed here is a **partial structural model**, not an exact VRPB formulation. It represents customer in-degree and out-degree, per-vehicle flow continuity, the one-route-per-vehicle depot degree limit, and the delivery-before-pickup rule by forbidding direct arcs from pickup customers to delivery customers. It does not encode vehicle capacity or eliminate disconnected subtours. Consequently, a low-energy binary assignment is not necessarily a feasible VRPB route plan.

No quantum simulator, quantum hardware, or additional optimization library is used. The instance has 128 arc-variable positions, so a dense statevector simulation would require storage proportional to $2^{128}$ amplitudes. The notebook therefore does not fabricate an optimization result, convergence history, speedup, noise measurement, or quantum advantage.

### Modeling assumptions

- Node 0 is the depot; delivery nodes are indexed first, followed by pickup nodes.
- The distance matrix gives the travel cost for each directed arc. The supplied matrix is symmetric, but the formulation uses directed arc variables.
- Each customer must have exactly one selected incoming arc and one selected outgoing arc across the modeled vehicles.
- Each vehicle has flow continuity at every customer and may use at most one departure from and one return to the depot.
- A route must serve deliveries before pickups. The encoded local rule disallows a direct pickup-to-delivery arc; by itself, it does not eliminate subtours or prove global route ordering.
- Demand and vehicle capacity are retained as instance data, but capacity is not enforced by this partial QUBO.

### Road map

1. Define the instance and QUBO data structures.
2. Build the quadratic objective and the stated penalty terms.
3. Describe the QAOA circuit resources implied by the QUBO.
4. Explain the QAOA cost and mixer operators, and why no optimization run is reported.
5. Inspect the QUBO structure and summarize its modeling limits.


## 1. From routing decisions to a QUBO

A QUBO (Quadratic Unconstrained Binary Optimization) model uses binary decision variables and expresses its objective as a polynomial of degree at most two. Constraints are represented by penalties added to the objective. A sufficiently large penalty makes violations costly, but a penalty model is only as complete as the constraints it actually includes.

Let $V=\{0,1,\ldots,n\}$ be the nodes, where node 0 is the depot, let $C=V\setminus\{0\}$ be the customers, and let $K$ be the vehicle set. For distinct nodes $i,j$ and vehicle $k$, define

$$
x_{ijk}=\begin{cases}
1,&\text{if vehicle }k\text{ traverses arc }i\to j,\\
0,&\text{otherwise.}
\end{cases}
$$

The travel-cost term is $\sum_{k\in K}\sum_{i\ne j}d_{ij}x_{ijk}$, where $d_{ij}$ is the distance from node $i$ to node $j$. The notebook stores the complete index range, including self-arc positions, but assigns a penalty to self-arcs and excludes them from the travel-cost term.

For a linear expression $g(x)=\sum_a c_ax_a+b$, the equality penalty $P g(x)^2$ is quadratic. For example, the customer in-degree condition uses $g(x)=\sum_{k\in K}\sum_{i\ne j}x_{ijk}-1$. The model uses this penalty for each customer's total in-degree and out-degree, and for per-vehicle flow continuity at each customer.

The resulting stored objective is written as

$$
E(x)=\frac{1}{2}x^{\mathsf T}Qx+\ell^{\mathsf T}x+c,
\qquad x_a\in\{0,1\}.
$$

Here $Q$ is a symmetric matrix of quadratic coefficients, $\ell$ is the vector of linear coefficients, and $c$ is a constant. The factor $\frac{1}{2}$ is part of the evaluation convention used by `QUBOFormulation.energy`. Equality penalties contribute both linear and quadratic terms after expansion. The code uses a penalty weight of 1000 for the modeled constraints; this is a modeling parameter, not a proof that all feasible solutions dominate all infeasible ones for every possible instance.

For QAOA, a binary variable is commonly mapped to a qubit operator using $x_a=(1-Z_a)/2$, where $Z_a$ is the Pauli-Z operator on qubit $a$. This converts the QUBO into a cost Hamiltonian. QAOA alternates cost evolution and a mixing operator, with parameters commonly written as $\boldsymbol{\gamma}=(\gamma_1,\ldots,\gamma_p)$ and $\boldsymbol{\beta}=(\beta_1,\ldots,\beta_p)$. A classical optimizer ordinarily updates these parameters using objective estimates from circuit measurements. This notebook explains that structure but does not execute those steps.


In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np

plt.style.use("seaborn-v0_8")

print("Core numerical and plotting libraries are ready.")

In [ ]:
@dataclass
class VRPBInstance:
    """Data for a VRPB instance with a depot indexed as node zero."""

    num_deliveries: int
    num_pickups: int
    delivery_demands: List[float]
    pickup_demands: List[float]
    capacity: float
    distance_matrix: np.ndarray

    def __post_init__(self) -> None:
        self.num_customers = self.num_deliveries + self.num_pickups
        self.delivery_indices = list(range(1, self.num_deliveries + 1))
        self.pickup_indices = list(
            range(self.num_deliveries + 1, self.num_customers + 1)
        )
        self.all_customer_indices = (
            self.delivery_indices + self.pickup_indices
        )

        expected_nodes = self.num_customers + 1
        if len(self.delivery_demands) != self.num_deliveries:
            raise ValueError("The number of delivery demands must match the delivery count.")
        if len(self.pickup_demands) != self.num_pickups:
            raise ValueError("The number of pickup demands must match the pickup count.")
        if self.distance_matrix.shape != (expected_nodes, expected_nodes):
            raise ValueError("The distance matrix must have one row and column per node.")
        if not np.all(np.isfinite(self.distance_matrix)):
            raise ValueError("The distance matrix must contain finite values.")
        if not np.allclose(self.distance_matrix, self.distance_matrix.T):
            raise ValueError("This instance assumes a symmetric distance matrix.")
        if not np.allclose(np.diag(self.distance_matrix), 0.0):
            raise ValueError("Travel distance from a node to itself must be zero.")


@dataclass
class QUBOFormulation:
    """Quadratic binary objective and the metadata needed to evaluate it."""

    Q_matrix: np.ndarray
    linear_terms: np.ndarray
    constant_term: float
    num_variables: int
    penalty_weights: Dict[str, float]
    constraint_satisfaction: Dict[str, float]

    def energy(self, solution: List[int]) -> float:
        """Evaluate the stored QUBO using its stated coefficient convention."""
        x = np.asarray(solution, dtype=float)
        if x.shape != (self.num_variables,):
            raise ValueError("The binary assignment has the wrong number of variables.")
        if not np.all(np.isin(x, [0.0, 1.0])):
            raise ValueError("Every QUBO variable must be either zero or one.")
        return float(
            0.5 * x.T @ self.Q_matrix @ x
            + self.linear_terms.T @ x
            + self.constant_term
        )


@dataclass
class QuantumCircuit:
    """Logical QAOA resource description; this is not an executed circuit."""

    num_qubits: int
    num_layers: int
    parameters: List[float]
    circuit_depth: int
    gate_counts: Dict[str, int]
    noise_model: Dict[str, float]


@dataclass
class QuantumOptimizer:
    """Container for results from an actual QAOA execution."""

    solution: List[int]
    objective_value: float
    execution_time: float
    quantum_advantage: float
    convergence_history: List[float]
    parameter_evolution: List[List[float]]
    noise_impact: float


print("VRPB, QUBO, circuit, and result data structures are defined.")

In [ ]:
def create_concrete_example() -> VRPBInstance:
    """Create the specified four-delivery, three-pickup example."""
    num_deliveries = 4
    num_pickups = 3
    delivery_demands = [5, 4, 6, 3]
    pickup_demands = [4, 5, 3]
    capacity = 20

    distance_matrix = np.array(
        [
            [0, 3, 4, 5, 6, 8, 7, 9],
            [3, 0, 2, 3, 4, 6, 5, 7],
            [4, 2, 0, 2, 3, 5, 4, 6],
            [5, 3, 2, 0, 2, 4, 3, 5],
            [6, 4, 3, 2, 0, 3, 2, 4],
            [8, 6, 5, 4, 3, 0, 2, 3],
            [7, 5, 4, 3, 2, 2, 0, 2],
            [9, 7, 6, 5, 4, 3, 2, 0],
        ],
        dtype=float,
    )

    return VRPBInstance(
        num_deliveries=num_deliveries,
        num_pickups=num_pickups,
        delivery_demands=delivery_demands,
        pickup_demands=pickup_demands,
        capacity=capacity,
        distance_matrix=distance_matrix,
    )


instance = create_concrete_example()
print(
    f"Instance: {instance.num_deliveries} deliveries, "
    f"{instance.num_pickups} pickups, and capacity {instance.capacity}."
)
print(f"Total delivery demand: {sum(instance.delivery_demands)}")
print(f"Total pickup demand: {sum(instance.pickup_demands)}")
print(f"Distance matrix shape: {instance.distance_matrix.shape}")

## 2. Constructing the partial QUBO

The formulation uses $x_{ijk}$ for the arc from node $i$ to node $j$ assigned to vehicle $k$. A customer in-degree penalty discourages customers from having anything other than one incoming selected arc; the out-degree penalty does the same for departures. Per-vehicle flow continuity links the number of arcs entering and leaving each customer. Pairwise penalties limit each vehicle to at most one departure and one return at the depot. Direct arcs from pickups to deliveries receive a large penalty to represent the local backhaul ordering rule.

These terms are quadratic or linear in the binary variables and can therefore be stored in a QUBO. The model deliberately does not add capacity or subtour-elimination terms. This boundary matters when interpreting any energy value: it is the objective of the partial model, not a certificate of a feasible VRPB solution.

In [ ]:
class QUBOFormulator:
    """Build a partial VRPB QUBO from arc costs and structural penalties."""

    def __init__(self, instance: VRPBInstance):
        self.instance = instance
        self.num_vehicles = 2
        self.penalty_weights = {
            "degree": 1000.0,
            "flow": 1000.0,
            "depot_degree": 1000.0,
            "precedence": 1000.0,
            "self_arc": 1000.0,
        }

    def formulate_qubo(self) -> QUBOFormulation:
        """Create the cost and penalty terms in a symmetric QUBO matrix."""
        num_nodes = self.instance.num_customers + 1
        num_variables = num_nodes * num_nodes * self.num_vehicles
        Q_matrix = np.zeros((num_variables, num_variables), dtype=float)
        linear_terms = np.zeros(num_variables, dtype=float)
        constant_term = 0.0

        def variable_index(i: int, j: int, k: int) -> int:
            return k * num_nodes * num_nodes + i * num_nodes + j

        # Travel costs occupy the linear part because each selected arc is binary.
        for k in range(self.num_vehicles):
            for i in range(num_nodes):
                for j in range(num_nodes):
                    index = variable_index(i, j, k)
                    if i == j:
                        linear_terms[index] += self.penalty_weights["self_arc"]
                    else:
                        linear_terms[index] += self.instance.distance_matrix[i, j]

        def add_squared_expression(
            coefficients: Dict[int, float],
            offset: float,
            weight: float,
        ) -> None:
            """Add weight * (sum(coefficients * x) + offset)^2."""
            nonlocal constant_term
            constant_term += weight * offset * offset

            for index, coefficient in coefficients.items():
                # Since x is binary, x squared equals x.
                linear_terms[index] += weight * (
                    coefficient * coefficient
                    + 2.0 * offset * coefficient
                )

            indices = list(coefficients)
            for position, first in enumerate(indices):
                for second in indices[position + 1 :]:
                    product_coefficient = (
                        2.0
                        * weight
                        * coefficients[first]
                        * coefficients[second]
                    )
                    Q_matrix[first, second] += product_coefficient
                    Q_matrix[second, first] += product_coefficient

        # Each customer has one incoming and one outgoing arc across all vehicles.
        for customer in self.instance.all_customer_indices:
            incoming = {
                variable_index(i, customer, k): 1.0
                for k in range(self.num_vehicles)
                for i in range(num_nodes)
                if i != customer
            }
            outgoing = {
                variable_index(customer, j, k): 1.0
                for k in range(self.num_vehicles)
                for j in range(num_nodes)
                if j != customer
            }
            add_squared_expression(incoming, -1.0, self.penalty_weights["degree"])
            add_squared_expression(outgoing, -1.0, self.penalty_weights["degree"])

        # Flow continuity prevents a vehicle from entering a customer without leaving it.
        for k in range(self.num_vehicles):
            for customer in self.instance.all_customer_indices:
                flow = {}
                for node in range(num_nodes):
                    if node != customer:
                        flow[variable_index(node, customer, k)] = 1.0
                        flow[variable_index(customer, node, k)] = -1.0
                add_squared_expression(flow, 0.0, self.penalty_weights["flow"])

        # Pairwise penalties enforce at most one depot departure and return per vehicle.
        for k in range(self.num_vehicles):
            depot_departures = [
                variable_index(0, customer, k)
                for customer in self.instance.all_customer_indices
            ]
            depot_returns = [
                variable_index(customer, 0, k)
                for customer in self.instance.all_customer_indices
            ]
            for arc_list in (depot_departures, depot_returns):
                for position, first in enumerate(arc_list):
                    for second in arc_list[position + 1 :]:
                        Q_matrix[first, second] += self.penalty_weights[
                            "depot_degree"
                        ]
                        Q_matrix[second, first] += self.penalty_weights[
                            "depot_degree"
                        ]

        # Forbid a direct transition from a pickup customer to a delivery customer.
        for pickup in self.instance.pickup_indices:
            for delivery in self.instance.delivery_indices:
                for k in range(self.num_vehicles):
                    index = variable_index(pickup, delivery, k)
                    linear_terms[index] += self.penalty_weights["precedence"]

        return QUBOFormulation(
            Q_matrix=Q_matrix,
            linear_terms=linear_terms,
            constant_term=constant_term,
            num_variables=num_variables,
            penalty_weights=self.penalty_weights.copy(),
            constraint_satisfaction={},
        )

    def _get_variable_index(
        self, i: int, j: int, k: int, num_nodes: int
    ) -> int:
        """Return the flattened index for arc variable x[i, j, k]."""
        return k * num_nodes * num_nodes + i * num_nodes + j


qubo_formulator = QUBOFormulator(instance)
qubo = qubo_formulator.formulate_qubo()

assert qubo.Q_matrix.shape == (qubo.num_variables, qubo.num_variables)
assert np.allclose(qubo.Q_matrix, qubo.Q_matrix.T)

print(f"QUBO variable positions: {qubo.num_variables}")
print(f"Q matrix shape: {qubo.Q_matrix.shape}")
print(f"Nonzero quadratic coefficients: {np.count_nonzero(qubo.Q_matrix)}")
print(f"Penalty weights: {qubo.penalty_weights}")

## 3. QAOA circuit design and execution boundary

QAOA is a hybrid quantum-classical approach. At depth $p$, a parameterized circuit alternates $p$ cost-evolution operators and $p$ mixing operators. The cost operator encodes the QUBO energy; the mixer changes the binary state. Measurements produce samples, from which an objective estimate can be calculated. A classical outer loop can then update the parameters $\boldsymbol{\gamma}$ and $\boldsymbol{\beta}$.

A QUBO with $N$ binary variables uses $N$ qubits in the direct encoding. The number of qubits alone does not determine practical feasibility: the cost Hamiltonian's interaction graph, circuit connectivity, gate decomposition, hardware noise, and measurement budget also matter. The circuit object below is only a logical resource description. It does not contain executable gates, a hardware noise simulation, or measured results.

For the current 128-variable representation, explicitly storing a general statevector would require $2^{128}$ complex amplitudes. Therefore, the optimizer class does not substitute random bit strings for quantum sampling or present a fabricated convergence trace. Its execution method raises a clear error until a real circuit execution backend is supplied.

In [ ]:
class QuantumCircuitDesigner:
    """Describe logical QAOA resources for a QUBO instance."""

    def __init__(self, qubo: QUBOFormulation):
        self.qubo = qubo
        self.num_qubits = qubo.num_variables

    def design_qaoa_circuit(self, num_layers: int = 3) -> QuantumCircuit:
        """Return a resource description for a depth-p QAOA circuit."""
        if num_layers < 1:
            raise ValueError("The number of QAOA layers must be positive.")

        # Standard QAOA has one gamma and one beta parameter per layer.
        parameters = [0.1] * (2 * num_layers)
        quadratic_interactions = int(np.count_nonzero(np.triu(self.qubo.Q_matrix, 1)))
        linear_interactions = int(np.count_nonzero(self.qubo.linear_terms))

        gate_counts = {
            "hadamard": self.num_qubits,
            "cost_terms": num_layers * (quadratic_interactions + linear_interactions),
            "mixer_rx": num_layers * self.num_qubits,
        }
        logical_layer_count = 1 + 2 * num_layers

        return QuantumCircuit(
            num_qubits=self.num_qubits,
            num_layers=num_layers,
            parameters=parameters,
            circuit_depth=logical_layer_count,
            gate_counts=gate_counts,
            noise_model={},
        )


class QAOAOptimizer:
    """Execution interface reserved for a real QAOA backend."""

    def __init__(self, qubo: QUBOFormulation):
        self.qubo = qubo
        self.circuit_designer = QuantumCircuitDesigner(qubo)

    def optimize(
        self, num_layers: int = 3, max_iterations: int = 50
    ) -> QuantumOptimizer:
        """Require a circuit backend rather than inventing optimization results."""
        if num_layers < 1:
            raise ValueError("The number of QAOA layers must be positive.")
        if max_iterations < 1:
            raise ValueError("The iteration limit must be positive.")
        raise NotImplementedError(
            "QAOA execution requires a quantum circuit backend and measured samples. "
            "This notebook provides the QUBO and logical circuit description only."
        )


circuit_designer = QuantumCircuitDesigner(qubo)
circuit = circuit_designer.design_qaoa_circuit(num_layers=3)
qaoa_optimizer = QAOAOptimizer(qubo)

print(f"Logical qubit count: {circuit.num_qubits}")
print(f"QAOA layers: {circuit.num_layers}")
print(f"Parameters (gamma and beta, one pair per layer): {len(circuit.parameters)}")
print(f"Logical layer count: {circuit.circuit_depth}")
print(f"Logical operation counts: {circuit.gate_counts}")
print("No quantum circuit has been executed.")

## 4. Inspecting the model

The matrix plot shows the locations of nonzero quadratic coefficients; it does not show a solution. The circuit chart reports logical operation counts derived from the stored QUBO. These counts are resource indicators, not hardware gate counts after compilation. The plot intentionally contains no convergence curve or speedup comparison because no optimization experiment has been run.

In [ ]:
def visualize_qubo_and_circuit(
    qubo: QUBOFormulation, circuit: QuantumCircuit
) -> None:
    """Plot QUBO sparsity and logical QAOA operation counts."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Partial VRPB QUBO and QAOA resource description", fontsize=14)

    matrix_image = ax1.imshow(
        qubo.Q_matrix != 0.0,
        interpolation="nearest",
        aspect="auto",
        cmap="Greys",
    )
    ax1.set_title("Nonzero entries in the quadratic matrix")
    ax1.set_xlabel("Variable index")
    ax1.set_ylabel("Variable index")
    fig.colorbar(matrix_image, ax=ax1, label="Nonzero coefficient")

    labels = list(circuit.gate_counts)
    counts = [circuit.gate_counts[label] for label in labels]
    ax2.bar(labels, counts, color="steelblue")
    ax2.set_title("Logical operation counts")
    ax2.set_xlabel("Operation category")
    ax2.set_ylabel("Count")
    ax2.tick_params(axis="x", rotation=25)

    fig.tight_layout()
    plt.show()


visualize_qubo_and_circuit(qubo, circuit)

## 5. Summary and interpretation

The notebook constructs a quadratic binary objective from travel costs and selected structural penalties, then relates the QUBO variables to the qubits and alternating operators used in QAOA. This gives a concrete view of the modeling sequence without implying that a circuit has been run.

### What the model represents

- Directed arc choices are represented by binary variables indexed by origin, destination, and vehicle.
- The objective includes travel distance, customer in-degree and out-degree penalties, vehicle flow-continuity penalties, depot degree penalties, pickup-to-delivery arc penalties, and self-arc penalties.
- The stored matrix is symmetric, and the energy is evaluated using $E(x)=\frac{1}{2}x^{\mathsf T}Qx+\ell^{\mathsf T}x+c$.
- The logical QAOA description uses one qubit per binary variable and two parameters per layer.

### What the model does not establish

The partial QUBO does not enforce vehicle capacity and does not eliminate disconnected subtours. Its local prohibition on direct pickup-to-delivery arcs is not, by itself, a complete proof that every selected route serves all deliveries before all pickups. Therefore, the model cannot certify a feasible VRPB route plan, and its energy cannot be interpreted as the cost of a validated solution.

The circuit description is not a quantum circuit execution. No quantum samples, optimized parameters, objective values, noise effects, convergence measurements, classical comparison, speedup, or quantum advantage are reported. Those claims require a complete model, an actual execution backend, and reproducible experimental measurements.

### Computational perspective

With $N$ binary variables, a direct statevector representation has $2^N$ complex amplitudes. This exponential state storage is a property of generic classical statevector simulation, not a proof that QAOA itself yields a speedup or that quantum hardware will solve this instance efficiently. Circuit depth, connectivity, noise, sampling, and the quality of the model all affect practical performance.

### Further modeling work

A complete VRPB QUBO would need to encode the omitted capacity and route-connectivity requirements while preserving the delivery-before-pickup rule. Penalty magnitudes would need to be assessed against the objective scale, and any experimental evaluation would need to report the circuit backend, parameter-update procedure, measurement budget, hardware or simulator settings, and feasibility checks. These are requirements for a fuller study, not results produced by this notebook.